# Rasing the Stakes
**Supplementary material for UMAP 2026 Submission**

Notebook for running Robustness Check for Mixed Effects Models

In [1]:
import sys
sys.path.append('../')

In [2]:
from pathlib import Path
import numpy as np

from pymer4.models import glmer

import pandas as pd
import polars as pl

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import metrics

In [3]:
from pymer4 import test_install
test_install()

All required R libraries found:
('lmerTest', 'emmeans', 'tidyverse', 'broom', 'broom.mixed', 'arrow', 'report')
Installation working successfully!


# Notebook and File Paths Setup

In [4]:
results_basepath = Path('data')
df_low = pd.read_csv(results_basepath / 'per_sample_results_low.csv')
df_high = pd.read_csv(results_basepath / 'per_sample_results_high.csv')
df_all = pd.concat([df_low, df_high], axis=0).reset_index(drop=True)

df_all["stakes_level"] = pd.Categorical(
    df_all["stakes_level"],
    categories=["low", "high"],  # first = reference
    ordered=True
)
df_all['time_taken_advised'] = df_all['time_taken_advised'] / 1000  # convert ms to s
df_all['z_time_taken_advised'] = (df_all['time_taken_advised'] - df_all['time_taken_advised'].mean()) / df_all['time_taken_advised'].std()

df_low = df_all[df_all['stakes_level'] == 'low'].copy()
df_high = df_all[df_all['stakes_level'] == 'high'].copy()

time_col = 'z_time_taken_advised'

In [5]:
user_AoRs_low = []
user_ids = df_low['part_id'].unique()
for i, uid in enumerate(user_ids):
    df_part = df_low[df_low['part_id'] == uid]
    AoR = metrics.calc_AoR(y_true=df_part['y_true'], 
                           y_model=df_part['y_pred'],
                           y_user_base=df_part['y_user_base'],
                           y_user_advised=df_part['y_user_advised'])
    user_AoRs_low.append(AoR)
user_AoRs_low = np.array(user_AoRs_low)
user_AoRs_low.shape

(24, 2)

In [6]:
user_AoRs_high = []
user_ids = df_high['part_id'].unique()
for i, uid in enumerate(user_ids):
    df_part = df_high[df_high['part_id'] == uid]
    AoR = metrics.calc_AoR(y_true=df_part['y_true'], 
                           y_model=df_part['y_pred'],
                           y_user_base=df_part['y_user_base'],
                           y_user_advised=df_part['y_user_advised'])
    user_AoRs_high.append(AoR)
user_AoRs_high = np.array(user_AoRs_high)
user_AoRs_high.shape

(28, 2)

In [7]:
def get_CAIR(df):
  cair_cases = (
        (df['y_user_base'] != df['y_true']) & 
        (df['y_pred'] == df['y_true']) & 
        (df['y_user_advised'] == df['y_pred'])
  )
  return cair_cases

def get_IAIR(df):
  iair_cases = (
        (df['y_user_base'] != df['y_true']) & 
        (df['y_pred'] == df['y_true']) & 
        (df['y_user_advised']!= df['y_pred'])
  )
  return iair_cases

def get_CSR(df):
  csr_cases = (
        (df['y_user_base'] == df['y_true']) & 
        (df['y_pred'] != df['y_true']) & 
        (df['y_user_advised'] == df['y_user_base'])
  )
  return csr_cases

def get_ISR(df):
  isr_cases = (
        (df['y_user_base'] == df['y_true']) & 
        (df['y_pred'] != df['y_true']) & 
        (df['y_user_advised'] != df['y_user_base'])
  )
  return isr_cases

In [8]:
df_all_cair = df_all[get_CAIR(df_all)].copy()
df_all_cair['CAIR'] = 1
df_all_iair = df_all[get_IAIR(df_all)].copy()
df_all_iair['CAIR'] = 0

df_all_comb_cair = pd.concat([df_all_cair, df_all_iair], axis=0, ignore_index=True)
df_all_comb_cair.head()

,part_id,blocky_id,stakes_level,y_true,y_pred,y_user_base,y_user_advised,time_taken_base,time_taken_advised,z_time_taken_advised,CAIR
0,study1-05,0575626d-1155-4f79-a8ed-151c82499cee,low,0,0,1,0,2596.0,3.797,-0.364812,1
1,study1-05,0a43a265-1afa-47a5-807e-402622e5b3c8,low,0,0,1,0,2010.0,4.397,-0.286653,1
2,study1-05,2fdd906b-a9b2-4c4b-b66b-94ab1efb1f97,low,1,1,0,1,3791.0,2.567,-0.525038,1
3,study1-05,6ee36074-0245-4135-b404-3158ecbfe6f5,low,1,1,0,1,4240.0,5.688,-0.118480,1
4,study1-05,a9560d62-361e-4028-b078-95afbe555b02,low,1,1,0,1,2483.0,3.702,-0.377187,1


# Mixed Effects Robustness Check

## Mixed-Effects Model of Decision Time on CAIR

In [9]:
df = df_all_comb_cair.copy()
df["CAIR"] = df["CAIR"].astype(int)
df["part_id"] = df["part_id"].astype("category")
df["blocky_id"] = df["blocky_id"].astype("category")

df_all_comb_pl = pl.DataFrame(df)

# GLMM logistic regression (logit link)
m_cair = glmer(
    f"CAIR ~ {time_col} * stakes_level + (1|part_id) + (1|blocky_id)",
    data=df_all_comb_pl,
    family="binomial"
)

gt = m_cair.fit(summary=True, exponentiate=True)

In [10]:
gt

GT(_tbl_data=shape: (8, 10)
┌────────────────┬──────────────────┬──────────┬──────────┬───┬───────────┬──────┬─────────┬───────┐
│ rfx            ┆ param            ┆ estimate ┆ conf_low ┆ … ┆ z_stat    ┆ df   ┆ p_value ┆ stars │
│ ---            ┆ ---              ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---  ┆ ---     ┆ ---   │
│ str            ┆ str              ┆ f64      ┆ f64      ┆   ┆ f64       ┆ f64  ┆ str     ┆ str   │
╞════════════════╪══════════════════╪══════════╪══════════╪═══╪═══════════╪══════╪═════════╪═══════╡
│ part_id-sd     ┆ (Intercept)      ┆ 0.851478 ┆ null     ┆ … ┆ null      ┆ null ┆ null    ┆ null  │
│ blocky_id-sd   ┆ (Intercept)      ┆ 0.383478 ┆ null     ┆ … ┆ null      ┆ null ┆ null    ┆ null  │
│ null           ┆ null             ┆ null     ┆ null     ┆ … ┆ null      ┆ null ┆ null    ┆ null  │
│ Fixed Effects: ┆ null             ┆ null     ┆ null     ┆ … ┆ null      ┆ null ┆ null    ┆ null  │
│ null           ┆ (Intercept)      ┆ 1.688025 ┆ 1.042849 ┆ … ┆ 2.130714  ┆ inf  ┆ 0.03311 ┆ *     │
│ null           ┆ z_time_taken_adv ┆ 0.584779 ┆ 0.347819 ┆ … ┆ -2.023985 ┆ inf  ┆ 0.04297 ┆ *     │
│                ┆ ised             ┆          ┆          ┆   ┆           ┆      ┆         ┆       │
│ null           ┆ stakes_levelhigh ┆ 1.421913 ┆ 0.764992 ┆ … ┆ 1.112956  ┆ inf  ┆ 0.2657  ┆       │
│ null           ┆ z_time_taken_adv ┆ 1.423199 ┆ 0.780364 ┆ … ┆ 1.151079  ┆ inf  ┆ 0.2497  ┆       │
│                ┆ ised:stakes_le…  ┆          ┆          ┆   ┆           ┆      ┆         ┆       │
└────────────────┴──────────────────┴──────────┴──────────┴───┴───────────┴──────┴─────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x182ece8c0>, _boxhead=Boxhead([ColInfo(var='rfx', type=<ColInfoTypeEnum.default: 1>, column_label='Random Effects:', column_align='left', column_width=None), ColInfo(var='param', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='estimate', type=<ColInfoTypeEnum.default: 1>, column_label='Estimate', column_align='right', column_width=None), ColInfo(var='conf_low', type=<ColInfoTypeEnum.default: 1>, column_label='CI-low', column_align='right', column_width=None), ColInfo(var='conf_high', type=<ColInfoTypeEnum.default: 1>, column_label='CI-high', column_align='right', column_width=None), ColInfo(var='std_error', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='z_stat', type=<ColInfoTypeEnum.default: 1>, column_label='Z-stat', column_align='right', column_width=None), ColInfo(var='df', type=<ColInfoTypeEnum.default: 1>, column_label='df', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x182ece380>, _spanners=Spanners([]), _heading=Heading(title='Formula: glmer(CAIR~z_time_taken_advised*stakes_level+(1|part_id)+(1|blocky_id))', subtitle=Md(text='Family: *binomial (link: *default*)*  \n            Number of observations: *543*  \n            Confidence intervals: *parametric*  \n            ---------------------  \n            Log-likelihood: *-337*  \n            AIC: *686* | BIC: *712*  \n            Residual error: *1.0*  \n        '), preheader=None), _stubhead=None, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['param'], rows=None, mask=None), grpname=None, colname='param', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=['param'], rows=None, mask=None), grpname=None, colname='param', rownum=1, colnum=None, styles=[Cell

In [11]:
df_all_csr = df_all[get_CSR(df_all)].copy()
df_all_csr['CSR'] = 1
df_all_isr = df_all[get_ISR(df_all)].copy()
df_all_isr['CSR'] = 0

df_all_comb_csr = pd.concat([df_all_csr, df_all_isr], axis=0, ignore_index=True)
df_all_comb_csr.head()

,part_id,blocky_id,stakes_level,y_true,y_pred,y_user_base,y_user_advised,time_taken_base,time_taken_advised,z_time_taken_advised,CSR
0,study1-05,0ec401e3-48a9-44e3-807c-e950d9f84760,low,0,1,0,0,2670.0,4.608,-0.259167,1
1,study1-05,1d690c7b-c566-482d-8940-2386ee5d5364,low,1,0,1,1,2128.0,2.800,-0.494686,1
2,study1-05,1fb1acfc-03c9-4a6e-b57a-e930051d04ca,low,1,0,1,1,2249.0,2.930,-0.477752,1
3,study1-05,22a11fec-8675-45c9-a24b-9a909bf9ea1d,low,1,0,1,1,2382.0,2.470,-0.537674,1
4,study1-05,66651fd4-ef75-4ce3-a1d2-255281116423,low,1,0,1,1,2400.0,2.812,-0.493123,1


## Mixed-Effects Model of Decision Time on CSR

In [12]:
df = df_all_comb_csr.copy()
df["CSR"] = df["CSR"].astype(int)
df["part_id"] = df["part_id"].astype("category")
df["blocky_id"] = df["blocky_id"].astype("category")

df_all_comb_pl = pl.DataFrame(df)

# GLMM logistic regression (logit link)
m_csr = glmer(
    f"CSR ~ {time_col} * stakes_level + (1|part_id) + (1|blocky_id)",
    data=df_all_comb_pl,
    family="binomial"
)

m_csr.fit(summary=True, exponentiate=True)

GT(_tbl_data=shape: (8, 10)
┌────────────────┬─────────────────┬──────────┬──────────┬───┬───────────┬──────┬──────────┬───────┐
│ rfx            ┆ param           ┆ estimate ┆ conf_low ┆ … ┆ z_stat    ┆ df   ┆ p_value  ┆ stars │
│ ---            ┆ ---             ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---  ┆ ---      ┆ ---   │
│ str            ┆ str             ┆ f64      ┆ f64      ┆   ┆ f64       ┆ f64  ┆ str      ┆ str   │
╞════════════════╪═════════════════╪══════════╪══════════╪═══╪═══════════╪══════╪══════════╪═══════╡
│ part_id-sd     ┆ (Intercept)     ┆ 1.287046 ┆ null     ┆ … ┆ null      ┆ null ┆ null     ┆ null  │
│ blocky_id-sd   ┆ (Intercept)     ┆ 0.779414 ┆ null     ┆ … ┆ null      ┆ null ┆ null     ┆ null  │
│ null           ┆ null            ┆ null     ┆ null     ┆ … ┆ null      ┆ null ┆ null     ┆ null  │
│ Fixed Effects: ┆ null            ┆ null     ┆ null     ┆ … ┆ null      ┆ null ┆ null     ┆ null  │
│ null           ┆ (Intercept)     ┆ 1.867297 ┆ 0.842275 ┆ … ┆ 1.537393  ┆ inf  ┆ 0.1242   ┆       │
│ null           ┆ z_time_taken_ad ┆ 4.030143 ┆ 1.400399 ┆ … ┆ 2.584377  ┆ inf  ┆ 0.009756 ┆ **    │
│                ┆ vised           ┆          ┆          ┆   ┆           ┆      ┆          ┆       │
│ null           ┆ stakes_levelhig ┆ 0.394408 ┆ 0.1635   ┆ … ┆ -2.07081  ┆ inf  ┆ 0.03838  ┆ *     │
│                ┆ h               ┆          ┆          ┆   ┆           ┆      ┆          ┆       │
│ null           ┆ z_time_taken_ad ┆ 0.246801 ┆ 0.080177 ┆ … ┆ -2.439046 ┆ inf  ┆ 0.01473  ┆ *     │
│                ┆ vised:stakes_le ┆          ┆          ┆   ┆           ┆      ┆          ┆       │
│                ┆ …               ┆          ┆          ┆   ┆           ┆      ┆          ┆       │
└────────────────┴─────────────────┴──────────┴──────────┴───┴───────────┴──────┴──────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x1830c1330>, _boxhead=Boxhead([ColInfo(var='rfx', type=<ColInfoTypeEnum.default: 1>, column_label='Random Effects:', column_align='left', column_width=None), ColInfo(var='param', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='estimate', type=<ColInfoTypeEnum.default: 1>, column_label='Estimate', column_align='right', column_width=None), ColInfo(var='conf_low', type=<ColInfoTypeEnum.default: 1>, column_label='CI-low', column_align='right', column_width=None), ColInfo(var='conf_high', type=<ColInfoTypeEnum.default: 1>, column_label='CI-high', column_align='right', column_width=None), ColInfo(var='std_error', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='z_stat', type=<ColInfoTypeEnum.default: 1>, column_label='Z-stat', column_align='right', column_width=None), ColInfo(var='df', type=<ColInfoTypeEnum.default: 1>, column_label='df', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x182edcfd0>, _spanners=Spanners([]), _heading=Heading(title='Formula: glmer(CSR~z_time_taken_advised*stakes_level+(1|part_id)+(1|blocky_id))', subtitle=Md(text='Family: *binomial (link: *default*)*  \n            Number of observations: *416*  \n            Confidence intervals: *parametric*  \n            ---------------------  \n            Log-likelihood: *-252*  \n            AIC: *516* | BIC: *540*  \n            Residual error: *1.0*  \n        '), preheader=None), _stubhead=None, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['param'], rows=None, mask=None), grpname=None, colname='param', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style='italic', weight=None